# ML-06 — Full Signal Audit for the Refresh Lane

This optional deeper audit supports the ML-07 rule. It uses the same warehouse unit and windows as ML-04: March 2026 features, April 2026 outcome, with February-to-March as a second-period check. June is not used. The future outcome is used only to test historical associations, never as a predictor.

## 1. Questions and audit method

1. **Visibility volume:** Does March impression volume track the next month's decline proxy? Volume is linked to the `quick-win` style of prioritizing opportunity, but may not be a stable decline-risk signal.
2. **CTR versus position:** Within comparable ranking bands, do low-CTR pages show a different next-month decline rate? This tests the idea behind CTR-fix flags while avoiding a raw position confound.

We first check distributions because search volume is heavy-tailed. Bucket tables include n and are treated as inconclusive for any bucket with fewer than 50 content items. Verdicts describe association only; they do not establish why performance changed or whether an edit would recover it.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
while not (repo_root / '.env').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / '.env').is_file():
    raise FileNotFoundError('Repository-root .env with HF_TOKEN is required; the token is never displayed.')
hf_token = None
for line in (repo_root / '.env').read_text(encoding='utf-8').splitlines():
    if '=' in line and not line.lstrip().startswith('#'):
        key, value = line.split('=', 1)
        if key.strip() == 'HF_TOKEN':
            hf_token = value.strip().strip(chr(34)).strip(chr(39))
            break
if not hf_token or not hf_token.startswith('hf_'):
    raise ValueError('HF_TOKEN is missing or malformed in .env; token value was not displayed.')
con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [hf_token])
del hf_token
warehouse = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
february_source = 'read_parquet(' + repr(warehouse + '/month=2026-02/*.parquet') + ')'
march_source = 'read_parquet(' + repr(warehouse + '/month=2026-03/*.parquet') + ')'
april_source = 'read_parquet(' + repr(warehouse + '/month=2026-04/*.parquet') + ')'
print('Warehouse connected using the ignored .env; no credential was printed.')

Warehouse connected using the ignored .env; no credential was printed.


In [2]:
march_april_sql = f'''
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar,
           SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks_mar,
           SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum_mar
    FROM {march_source}
    GROUP BY client_hash_id, content_hash_id
), april AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_apr,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_apr
    FROM {april_source}
    GROUP BY client_hash_id, content_hash_id
)
SELECT march.client_hash_id, march.content_hash_id,
       march.impressions_mar, march.clicks_mar,
       march.position_sum_mar / NULLIF(march.impressions_mar, 0) AS avg_position_mar,
       100.0 * march.clicks_mar / NULLIF(march.impressions_mar, 0) AS ctr_mar,
       CAST(april.impressions_apr <= 0.8 * march.impressions_mar AS INTEGER) AS april_decline_label
FROM march INNER JOIN april USING (client_hash_id, content_hash_id)
WHERE march.gsc_days_mar >= 20 AND april.gsc_days_apr >= 20
  AND march.impressions_mar > 0
'''
audit_items = con.execute(march_april_sql).df()
LABEL = 'april_decline_label'
print(f'Paired, adequately observed content items: {len(audit_items):,}')
print(f'April decline-proxy base rate: {audit_items[LABEL].mean():.3f}')
assert len(audit_items) == 95633

Paired, adequately observed content items: 95,633
April decline-proxy base rate: 0.495


## 2. Distributions before verdicts

Impressions and clicks are long-tailed. The quantiles and maximum make clear why averages alone are misleading. No raw identifiers are displayed.

In [3]:
distribution_rows = []
for column in ['impressions_mar', 'clicks_mar', 'ctr_mar', 'avg_position_mar']:
    values = audit_items[column].dropna()
    distribution_rows.append({
        'metric': column,
        'n': len(values),
        'median': values.median(),
        'p90': values.quantile(0.90),
        'p99': values.quantile(0.99),
        'max': values.max(),
        'skew': values.skew(),
    })
distribution_table = pd.DataFrame(distribution_rows)
display(distribution_table.round(2))
print('Impressions max / p99:', round(distribution_table.loc[distribution_table['metric'].eq('impressions_mar'), 'max'].iloc[0] / distribution_table.loc[distribution_table['metric'].eq('impressions_mar'), 'p99'].iloc[0], 2))

,metric,n,median,p90,p99,max,skew
0,impressions_mar,95633,820.00,6730.80,30076.36,617124.00,15.31
1,clicks_mar,95633,1.00,20.00,106.00,5668.00,56.42
2,ctr_mar,95633,0.11,0.67,1.89,16.25,6.40
3,avg_position_mar,95633,8.61,35.45,70.80,106.89,1.91


Impressions max / p99: 20.52


## 3. Signal test: visibility volume

Claim: higher March impression volume should identify pages with useful visibility, but it may not monotonically predict decline. The April decline label is only the outcome used to test the claim.

In [4]:
audit_items['volume_bucket'] = pd.cut(
    audit_items['impressions_mar'],
    bins=[0, 100, 500, 2000, np.inf],
    labels=['<100', '100-499', '500-1,999', '2,000+'],
    right=False,
    include_lowest=True,
)
volume_table = (
    audit_items.groupby('volume_bucket', observed=False)
    .agg(n=(LABEL, 'size'), april_decline_rate=(LABEL, 'mean'))
    .reset_index()
)
display(volume_table.assign(april_decline_rate=volume_table['april_decline_rate'].round(3)))
assert volume_table['n'].min() >= 50
print('Verdict: MIXED — decline is much less common below 100 impressions, rises in the middle buckets, then eases in the 2,000+ bucket; volume alone is not a consistent risk ordering.')

,volume_bucket,n,april_decline_rate
0,<100,6692,0.254
1,100-499,30137,0.518
2,"500-1,999",29575,0.544
3,"2,000+",29229,0.477


Verdict: MIXED — decline is much less common below 100 impressions, rises in the middle buckets, then eases in the 2,000+ bucket; volume alone is not a consistent risk ordering.


## 4. Signal test: CTR within position bands

Claim: low CTR is associated with next-month decline even when pages are compared within broad position bands. CTR is clicks divided by impressions, in percent; it is not averaged across daily rows.

In [5]:
audit_items['position_band'] = np.select(
    [audit_items['avg_position_mar'] <= 10, audit_items['avg_position_mar'] <= 20],
    ['position 1-10', 'position 11-20'],
    default='outside position 1-20',
)
audit_items['ctr_bucket'] = np.where(audit_items['ctr_mar'] < 0.5, 'CTR <0.5%', 'CTR >=0.5%')
audit_items['ctr_position_bucket'] = np.where(
    audit_items['position_band'].eq('outside position 1-20'),
    audit_items['position_band'],
    audit_items['position_band'] + ' / ' + audit_items['ctr_bucket'],
)
ctr_table = (
    audit_items.groupby('ctr_position_bucket', observed=True)
    .agg(n=(LABEL, 'size'), april_decline_rate=(LABEL, 'mean'))
    .reset_index()
    .sort_values('ctr_position_bucket')
)
display(ctr_table.assign(april_decline_rate=ctr_table['april_decline_rate'].round(3)))
assert ctr_table['n'].min() >= 50
within_bands = ctr_table.loc[ctr_table['ctr_position_bucket'].str.startswith('position')].copy()
print('Verdict: CONFIRMED (association) — within both position bands, the low-CTR bucket has a higher April decline rate. This supports a review flag, not a causal explanation.')

,ctr_position_bucket,n,april_decline_rate
0,outside position 1-20,23906,0.464
1,position 1-10 / CTR <0.5%,42398,0.553
2,position 1-10 / CTR >=0.5%,10378,0.319
3,position 11-20 / CTR <0.5%,16008,0.522
4,position 11-20 / CTR >=0.5%,2943,0.373


Verdict: CONFIRMED (association) — within both position bands, the low-CTR bucket has a higher April decline rate. This supports a review flag, not a causal explanation.


## 5. Repeat the volume test on another period

February features and March outcome are a second mid-panel check (not the June sealed month). The same thresholds and minimum-coverage rule are applied.

In [6]:
february_march_sql = f'''
WITH february AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_feb,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_feb
    FROM {february_source}
    GROUP BY client_hash_id, content_hash_id
), march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar
    FROM {march_source}
    GROUP BY client_hash_id, content_hash_id
)
SELECT february.impressions_feb,
       CAST(march.impressions_mar <= 0.8 * february.impressions_feb AS INTEGER) AS march_decline_label
FROM february INNER JOIN march USING (client_hash_id, content_hash_id)
WHERE february.gsc_days_feb >= 20 AND march.gsc_days_mar >= 20
  AND february.impressions_feb > 0
'''
repeat_items = con.execute(february_march_sql).df()
repeat_items['volume_bucket'] = pd.cut(
    repeat_items['impressions_feb'],
    bins=[0, 100, 500, 2000, np.inf],
    labels=['<100', '100-499', '500-1,999', '2,000+'],
    right=False,
    include_lowest=True,
)
repeat_table = (
    repeat_items.groupby('volume_bucket', observed=False)
    .agg(n=('march_decline_label', 'size'), march_decline_rate=('march_decline_label', 'mean'))
    .reset_index()
)
print(f'February-March eligible items: {len(repeat_items):,}')
display(repeat_table.assign(march_decline_rate=repeat_table['march_decline_rate'].round(3)))
assert repeat_table['n'].min() >= 50
print('Repeat verdict: MIXED — low volume remains the lowest-risk bucket; the upper buckets are not strictly monotonic.')
con.close()

February-March eligible items: 71,497


,volume_bucket,n,march_decline_rate
0,<100,7027,0.087
1,100-499,22918,0.197
2,"500-1,999",21846,0.184
3,"2,000+",19706,0.159


Repeat verdict: MIXED — low volume remains the lowest-risk bucket; the upper buckets are not strictly monotonic.


## 6. Audit conclusion and limitations

- **Volume — MIXED:** it signals available opportunity, but does not steadily increase decline risk. Use it as a floor for editorial value, not as a standalone risk score.
- **CTR within position — CONFIRMED association:** low CTR aligns with higher next-month decline rates in both position groups. It justifies checking the snippet and page/query match; it does not say the page is stale.
- **Second period:** February-to-March repeats the mixed volume pattern, so the finding is not unique to March-to-April.
- **Limitations:** each paired item must have at least 20 GSC-observed days per month and positive feature-month impressions. The observed panel is selected by data coverage; monthly aggregation hides daily volatility, query mix, SERP features, intent, and page freshness. The decline proxy is not evidence that a refresh causes recovery.

These findings support the small, readable ML-07 baseline and leave room for a future model to demonstrate improvement. No names, URLs, or raw query text are included.